# Manitoba2 — Data Processing

Goal: produce (1) **restated municipality-year yields**, (2) **nonlinear weather index feature sets** for **station** and **reanalysis** data, and (3) two final merged modeling datasets.

Outputs are saved to `data/prepared/`.

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path("..").resolve()
DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PREP = REPO_ROOT / "data" / "prepared"
DATA_PREP.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

MONTH_MAP = {5:"May", 6:"Jun", 7:"Jul", 8:"Aug", 9:"Sep", 10:"Oct"}
GROWING_MONTHS = [5,6,7,8,9,10]
YEAR_MIN, YEAR_MAX = 1996, 2011


## 1) Load raw data

In [10]:
yields = pd.read_csv(DATA_RAW / "Yields.csv")
w_obs = pd.read_csv(DATA_RAW / "Weather Obs.csv")
w_rea = pd.read_csv(DATA_RAW / "Weather Reanalysis.csv")

print("Yields:", yields.shape)
print("Weather Obs:", w_obs.shape)
print("Weather Reanalysis:", w_rea.shape)

display(yields.head())
display(w_obs.head())
display(w_rea.head())


Yields: (44448, 9)
Weather Obs: (233738, 13)
Weather Reanalysis: (490896, 9)


,Year,Municipality,Crop,Variety,Farms,Acres,Yield/Acre,t,i
0,1996,ALEXANDER,ARGENTINE CANOLA,INNOVATOR (HCN 92) (LT),3,575.0,0.579,1,1
1,1996,ALEXANDER,ARGENTINE CANOLA,QUANTUM (91-21864 NA),6,1125.0,0.780,1,1
2,1996,ALEXANDER,ARGENTINE CANOLA,45A71 (NS1471)(ST),4,1012.0,0.756,1,1
3,1996,ALEXANDER,BARLEY,ROBUST,9,1247.0,1.607,1,1
4,1996,ALEXANDER,CANARYSEED,KEET,3,535.0,0.184,1,1


,Municipality,Date/Time,Year,Month,Day,Max_Temp,Min_Temp,Mean_Temp,HDD,CDD,Total Rain (mm),Total Snow,Ptol
0,ALEXANDER,1/1/96,1996,1,1,-10.0,-18.0,-14.00,32.0,0.0,0.0,4.0,4.0
1,ALEXANDER,1/2/96,1996,1,2,-11.0,-21.0,-16.00,34.0,0.0,0.0,2.0,2.0
2,ALEXANDER,1/3/96,1996,1,3,-14.0,-22.0,-18.00,36.0,0.0,0.0,1.0,1.0
3,ALEXANDER,1/4/96,1996,1,4,-26.5,-36.0,-31.25,49.3,0.0,0.0,0.0,0.0
4,ALEXANDER,1/5/96,1996,1,5,-30.0,-39.0,-34.50,52.5,0.0,0.0,0.0,0.0


,Municipality,Date/Time,Year,Month,Day,Max_Temp,Min_Temp,Mean_Temp,Ptol
0,ALEXANDER,1/1/96,1996,1,1,-10.416400,-21.008989,-15.712695,-1.370000e-10
1,ALEXANDER,1/2/96,1996,1,2,-15.238113,-18.910885,-17.074499,-1.370000e-10
2,ALEXANDER,1/3/96,1996,1,3,-13.913316,-21.768749,-17.841032,1.088851e-03
3,ALEXANDER,1/4/96,1996,1,4,-22.517648,-28.702039,-25.609844,-1.370000e-10
4,ALEXANDER,1/5/96,1996,1,5,-23.918762,-30.488448,-27.203605,-1.370000e-10


## 2) Yield processing — crop mix restatement (municipality-level adaptation)
We build **municipality-year** restated yields using a benchmark crop mix (last 5 years) and **cubic polynomial smoothing** of crop acreage shares over time.

In [11]:
# Basic cleaning / filtering
y = yields.copy()
y.columns = [c.strip() for c in y.columns]

# Ensure expected columns exist
expected = {"Year","Municipality","Crop","Variety","Farms","Acres","Yield/Acre"}
missing = expected - set(y.columns)
if missing:
    raise ValueError(f"Missing columns in Yields.csv: {missing}")

# Filter years
y = y[(y["Year"]>=YEAR_MIN) & (y["Year"]<=YEAR_MAX)].copy()

# Make numeric
for col in ["Farms","Acres","Yield/Acre"]:
    y[col] = pd.to_numeric(y[col], errors="coerce")

# Drop rows with no acres or yield
y = y[(y["Acres"]>0) & (y["Yield/Acre"].notna())].copy()

# 2.1) Aggregate varieties -> crop-level yield per municipality-year (weighted by acres)
crop_year = (
    y.groupby(["Municipality","Year","Crop"], as_index=False)
     .apply(lambda g: pd.Series({
         "Acres_crop": g["Acres"].sum(),
         "Yield_crop": np.average(g["Yield/Acre"], weights=g["Acres"])
     }))
     .reset_index(drop=True)
)

# 2.2) Total acres per municipality-year and crop shares
tot = crop_year.groupby(["Municipality","Year"], as_index=False)["Acres_crop"].sum().rename(columns={"Acres_crop":"Acres_total"})
crop_year = crop_year.merge(tot, on=["Municipality","Year"], how="left")
crop_year["Share"] = crop_year["Acres_crop"] / crop_year["Acres_total"]

display(crop_year.head())
print("Municipalities:", crop_year["Municipality"].nunique(), "Years:", crop_year["Year"].nunique())


C:\Users\henri\AppData\Local\Temp\ipykernel_16268\1564955564.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,Municipality,Year,Crop,Acres_crop,Yield_crop,Acres_total,Share
0,ALEXANDER,1996,ARGENTINE CANOLA,2712.0,0.728428,11997.0,0.226057
1,ALEXANDER,1996,BARLEY,1247.0,1.607000,11997.0,0.103943
2,ALEXANDER,1996,CANARYSEED,535.0,0.184000,11997.0,0.044594
3,ALEXANDER,1996,OATS,659.0,1.215000,11997.0,0.054930
4,ALEXANDER,1996,POLISH CANOLA,1043.0,0.278000,11997.0,0.086938


Municipalities: 99 Years: 16


### 2.3) Determine benchmark crop mix per municipality (last 5 years, cover ≥90% acres)
For each municipality, we compute acres over the last 5 years and pick the minimal set of crops covering 90% of acres. All remaining crops are grouped as **Others**.

In [12]:
def main_crop_mix_for_municipality(df_muni: pd.DataFrame, coverage=0.90, last_n_years=5):
    # df_muni has Municipality, Year, Crop, Acres_crop
    max_year = df_muni["Year"].max()
    recent = df_muni[df_muni["Year"] >= (max_year - last_n_years + 1)]
    acres_by_crop = recent.groupby("Crop")["Acres_crop"].sum().sort_values(ascending=False)
    if acres_by_crop.empty:
        return [], max_year
    cum = acres_by_crop.cumsum() / acres_by_crop.sum()
    main = list(cum[cum <= coverage].index)
    # Ensure at least one crop; also include the crop that crosses threshold
    if len(main) < len(acres_by_crop):
        # include first crop after threshold
        first_after = cum.index[len(main)]
        main = main + [first_after] if first_after not in main else main
    return main, max_year

muni_main = {}
for muni, g in crop_year.groupby("Municipality"):
    main, maxy = main_crop_mix_for_municipality(g, coverage=0.90, last_n_years=5)
    muni_main[muni] = {"main_crops": main, "max_year": maxy}

# quick peek
sample_muni = list(muni_main.keys())[0]
sample_muni, muni_main[sample_muni]["main_crops"][:10], len(muni_main[sample_muni]["main_crops"])


('ALEXANDER',
 ['ARGENTINE CANOLA', 'SOYBEANS', 'OATS', 'RED SPRING WHEAT', 'BARLEY'],
 5)

### 2.4) Build 'Others' crop group and time-series shares; fit cubic polynomials
We fit, for each municipality and each crop in (main mix + Others), a cubic polynomial of share vs year:

$a^R_{i,k,t} = \alpha + \beta t + \gamma t^2 + \eta t^3$

Then we clip negatives to 0 and renormalize shares to sum to 1 per municipality-year.

In [13]:
def fit_cubic_shares(years, shares):
    # years: array, shares: array
    x = np.asarray(years, dtype=float)
    yv = np.asarray(shares, dtype=float)
    # center/scale for numerical stability
    x0 = x.mean()
    xs = x.std() if x.std() > 0 else 1.0
    z = (x - x0) / xs
    # polyfit degree 3 (handles full length; shares can contain zeros)
    coef = np.polyfit(z, yv, deg=3)
    pred = np.polyval(coef, z)
    return pred

# Create full grid of municipality x year
years_all = np.arange(YEAR_MIN, YEAR_MAX+1)
munis = crop_year["Municipality"].unique()

restated_rows = []  # will store muni-year-crop (including Others) with restated shares and yields
for muni in munis:
    main = set(muni_main[muni]["main_crops"])
    g = crop_year[crop_year["Municipality"]==muni].copy()

    # Build crop yield table (main crops and non-main)
    # We'll compute per year:
    # - Yield_crop and Acres_crop for each crop
    # - Others: weighted avg yield over non-main crops
    # - Shares for main crops and Others
    for yr in years_all:
        gy = g[g["Year"]==yr]
        if gy.empty:
            continue

    # Prepare wide arrays for shares across years for main crops
    shares_by_crop = {}
    yield_by_crop_year = {}

    # Precompute per year crop yields (main crops) and per year Others yield
    for yr in years_all:
        gy = g[g["Year"]==yr]
        if gy.empty:
            continue

        # yields for each crop
        for _, r in gy.iterrows():
            yield_by_crop_year[(r["Crop"], yr)] = r["Yield_crop"]

        # Others yield: all non-main crops in that year
        non_main = gy[~gy["Crop"].isin(main)]
        if len(non_main) > 0:
            others_y = np.average(non_main["Yield_crop"], weights=non_main["Acres_crop"])
            yield_by_crop_year[("Others", yr)] = others_y
        else:
            yield_by_crop_year[("Others", yr)] = np.nan

    # Shares for main crops per year; missing -> 0
    for crop in list(main):
        s = []
        yyrs = []
        for yr in years_all:
            gy = g[g["Year"]==yr]
            if gy.empty:
                continue
            yyrs.append(yr)
            row = gy[gy["Crop"]==crop]
            s.append(float(row["Share"].iloc[0]) if len(row) else 0.0)
        if yyrs:
            shares_by_crop[crop] = (np.array(yyrs), np.array(s))

    # Others share per year
    yyrs = []
    others_s = []
    for yr in years_all:
        gy = g[g["Year"]==yr]
        if gy.empty:
            continue
        yyrs.append(yr)
        s_main = gy[gy["Crop"].isin(main)]["Share"].sum()
        others_s.append(float(max(0.0, 1.0 - s_main)))
    if yyrs:
        shares_by_crop["Others"] = (np.array(yyrs), np.array(others_s))

    # Fit cubic for each crop group
    fitted = {}
    for crop, (yrs, s) in shares_by_crop.items():
        pred = fit_cubic_shares(yrs, s)
        fitted[crop] = pd.Series(pred, index=yrs)

    # Assemble restated shares per year and renormalize
    for yr in yyrs:
        crops_here = list(fitted.keys())
        vals = np.array([float(fitted[c].get(yr, 0.0)) for c in crops_here], dtype=float)
        vals = np.clip(vals, 0.0, None)
        ssum = vals.sum()
        if ssum <= 0:
            # fallback to original shares (main + Others)
            # compute original shares
            gy = g[g["Year"]==yr]
            orig = []
            for c in crops_here:
                if c == "Others":
                    orig.append(float(max(0.0, 1.0 - gy[gy["Crop"].isin(main)]["Share"].sum())))
                else:
                    row = gy[gy["Crop"]==c]
                    orig.append(float(row["Share"].iloc[0]) if len(row) else 0.0)
            vals = np.array(orig, dtype=float)
            vals = np.clip(vals, 0.0, None)
            ssum = vals.sum() if vals.sum() > 0 else 1.0
        vals = vals / ssum

        for c, sh in zip(crops_here, vals):
            restated_rows.append({
                "Municipality": muni,
                "Year": int(yr),
                "CropGroup": c,
                "Share_restated": float(sh),
                "Yield_cropgroup": float(yield_by_crop_year.get((c, yr), np.nan))
            })

restated = pd.DataFrame(restated_rows)

# For years where Others yield is NaN (no non-main crops), set it to 0 and share should be near 0 anyway
restated["Yield_cropgroup"] = restated["Yield_cropgroup"].fillna(0.0)

# Compute restated yield per municipality-year
y_rest = (
    restated.assign(weighted=lambda d: d["Share_restated"]*d["Yield_cropgroup"])
            .groupby(["Municipality","Year"], as_index=False)["weighted"].sum()
            .rename(columns={"weighted":"Yield_restated"})
)

display(y_rest.head())
print("Restated yield rows:", y_rest.shape)

# Save
y_rest.to_csv(DATA_PREP / "yields_restated_muni_year.csv", index=False)


,Municipality,Year,Yield_restated
0,ALEXANDER,1996,0.907619
1,ALEXANDER,1997,0.859800
2,ALEXANDER,1998,0.938502
3,ALEXANDER,1999,1.013855
4,ALEXANDER,2000,0.713670


Restated yield rows: (1565, 3)


## 3) Weather index feature engineering
We create **nonlinear daily indices** (NGDL, DGDL, DGDH, PREH, PREL) then aggregate to monthly and May–Oct seasonal features.

Precip thresholds λ1 and λ2 are computed as empirical **25% and 75% quantiles** of daily precipitation (Ptol) during May–Oct.

In [14]:
def prepare_weather(df: pd.DataFrame, is_obs: bool):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    # Rename precipitation
    if is_obs:
        # obs has both Total Rain and Ptol; use Ptol for consistency
        pass
    # Ensure expected cols
    needed = {"Municipality","Year","Month","Day","Max_Temp","Min_Temp","Mean_Temp","Ptol"}
    miss = needed - set(df.columns)
    if miss:
        raise ValueError(f"Missing columns in weather data: {miss}")
    # filter years/months
    df = df[(df["Year"]>=YEAR_MIN) & (df["Year"]<=YEAR_MAX)].copy()
    df = df[df["Month"].isin(GROWING_MONTHS)].copy()
    # numeric
    for c in ["Max_Temp","Min_Temp","Mean_Temp","Ptol","Year","Month","Day"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["Municipality","Year","Month","Max_Temp","Min_Temp","Ptol"])
    return df

def compute_precip_thresholds(df):
    # empirical quartiles on daily Ptol in growing season
    q1 = float(df["Ptol"].quantile(0.25))
    q3 = float(df["Ptol"].quantile(0.75))
    return q1, q3

theta1 = [0,1,2,3,4]      # for NGDL (night)
theta2 = [6,7,8,9,10]     # for DGDL (day low)
theta3 = [26,27,28,29,30] # for DGDH (day high)

def add_daily_indices(df, lam1, lam2):
    df = df.copy()
    # base
    df["Tmax"] = df["Max_Temp"]
    df["Tmin"] = df["Min_Temp"]
    df["Tavg"] = df["Mean_Temp"]
    df["P"] = df["Ptol"]
    # indices (note: NGDL, DGDL are <=0; DGDH >=0)
    for i, th in enumerate(theta1, start=1):
        df[f"NGDL{i}"] = np.minimum(df["Tmin"] - th, 0.0)
    for i, th in enumerate(theta2, start=1):
        df[f"DGDL{i}"] = np.minimum(df["Tmax"] - th, 0.0)
    for i, th in enumerate(theta3, start=1):
        df[f"DGDH{i}"] = np.maximum(df["Tmax"] - th, 0.0)
    df["PREH"] = np.maximum(df["P"] - lam1, 0.0)
    df["PREL"] = np.minimum(df["P"] - lam2, 0.0)
    return df

def aggregate_features(df):
    # df is municipality-daily in growing season, already has indices
    df = df.copy()
    # monthly aggregates
    idx_cols = [c for c in df.columns if c in ["Tmax","Tmin","Tavg","P"] or c.startswith(("NGDL","DGDL","DGDH","PREH","PREL"))]
    group_keys = ["Municipality","Year","Month"]

    agg = {}
    for c in idx_cols:
        if c in ["Tmax","Tmin","Tavg","P"]:
            agg[c] = ["max","min","mean"]
        else:
            agg[c] = ["max","min","mean", lambda s: (s != 0).sum()]
    monthly = df.groupby(group_keys).agg(agg)
    # rename columns
    monthly.columns = [
        f"{col}_{('avg' if func=='mean' else func) if func!='<lambda>' else 'cot'}"
        for col, func in monthly.columns.to_flat_index()
    ]
    monthly = monthly.reset_index()

    # reshape to wide with month suffix
    out_parts = []
    for m in GROWING_MONTHS:
        mm = monthly[monthly["Month"]==m].copy()
        suffix = "_" + MONTH_MAP[m]
        feat_cols = [c for c in mm.columns if c not in ["Municipality","Year","Month"]]
        mm = mm.drop(columns=["Month"])
        mm = mm.rename(columns={c: c + suffix for c in feat_cols})
        out_parts.append(mm)
    wide_month = out_parts[0]
    for part in out_parts[1:]:
        wide_month = wide_month.merge(part, on=["Municipality","Year"], how="outer")

    # seasonal (May-Oct) aggregates
    group_keys2 = ["Municipality","Year"]
    seasonal = df.groupby(group_keys2).agg(agg)
    seasonal.columns = [
        f"{col}_{('avg' if func=='mean' else func) if func!='<lambda>' else 'cot'}_GS"
        for col, func in seasonal.columns.to_flat_index()
    ]
    seasonal = seasonal.reset_index()

    # merge monthly+seasonal
    feats = wide_month.merge(seasonal, on=["Municipality","Year"], how="left")
    return feats

def build_weather_feature_table(raw_df, is_obs: bool, out_name: str):
    w = prepare_weather(raw_df, is_obs=is_obs)
    lam1, lam2 = compute_precip_thresholds(w)
    print(f"{out_name}: precip thresholds (q25,q75) = ({lam1:.4f}, {lam2:.4f})")
    w = add_daily_indices(w, lam1, lam2)
    feats = aggregate_features(w)
    # drop constant & duplicates
    key_cols = ["Municipality","Year"]
    # drop constant
    feature_cols = [c for c in feats.columns if c not in key_cols]
    nun = feats[feature_cols].nunique(dropna=False)
    feature_cols = [c for c in feature_cols if nun[c] > 1]
    feats = feats[key_cols + feature_cols]
    # drop duplicates by hash
    seen = {}
    keep = []
    for c in feature_cols:
        h = pd.util.hash_pandas_object(feats[c], index=False).sum()
        if h not in seen:
            seen[h] = c
            keep.append(c)
    feats = feats[key_cols + keep]
    feats.to_csv(DATA_PREP / out_name, index=False)
    return feats

feat_station = build_weather_feature_table(w_obs, is_obs=True, out_name="weather_idx_station_muni_year.csv")
feat_reanalysis = build_weather_feature_table(w_rea, is_obs=False, out_name="weather_idx_reanalysis_muni_year.csv")

print("Station features:", feat_station.shape)
print("Reanalysis features:", feat_reanalysis.shape)
display(feat_station.head())


weather_idx_station_muni_year.csv: precip thresholds (q25,q75) = (0.0000, 1.0000)
weather_idx_reanalysis_muni_year.csv: precip thresholds (q25,q75) = (0.0000, 0.0000)
Station features: (628, 425)
Reanalysis features: (1344, 378)


,Municipality,Year,Tmax_max_May,Tmax_min_May,Tmax_avg_May,Tmin_max_May,Tmin_min_May,Tmin_avg_May,Tavg_max_May,Tavg_min_May,Tavg_avg_May,P_max_May,P_min_May,P_avg_May,NGDL1_min_May,NGDL1_avg_May,NGDL1_<lambda_0>_May,NGDL2_max_May,NGDL2_min_May,NGDL2_avg_May,NGDL2_<lambda_0>_May,NGDL3_max_May,NGDL3_min_May,NGDL3_avg_May,NGDL3_<lambda_0>_May,NGDL4_max_May,NGDL4_min_May,NGDL4_avg_May,NGDL4_<lambda_0>_May,NGDL5_max_May,NGDL5_min_May,NGDL5_avg_May,NGDL5_<lambda_0>_May,DGDL1_min_May,DGDL1_avg_May,DGDL1_<lambda_0>_May,DGDL2_min_May,DGDL2_avg_May,DGDL2_<lambda_0>_May,DGDL3_min_May,DGDL3_avg_May,DGDL3_<lambda_0>_May,DGDL4_min_May,DGDL4_avg_May,DGDL4_<lambda_0>_May,DGDL5_min_May,DGDL5_avg_May,DGDL5_<lambda_0>_May,DGDH1_max_May,DGDH1_avg_May,DGDH1_<lambda_0>_May,DGDH2_max_May,DGDH2_avg_May,DGDH2_<lambda_0>_May,DGDH3_max_May,DGDH3_avg_May,DGDH3_<lambda_0>_May,DGDH4_max_May,DGDH4_avg_May,DGDH4_<lambda_0>_May,...,Tmax_max_GS,Tmax_min_GS,Tmax_avg_GS,Tmin_max_GS,Tmin_min_GS,Tmin_avg_GS,Tavg_max_GS,Tavg_min_GS,Tavg_avg_GS,P_max_GS,P_avg_GS,NGDL1_min_GS,NGDL1_avg_GS,NGDL1_<lambda_0>_GS,NGDL2_min_GS,NGDL2_avg_GS,NGDL2_<lambda_0>_GS,NGDL3_min_GS,NGDL3_avg_GS,NGDL3_<lambda_0>_GS,NGDL4_min_GS,NGDL4_avg_GS,NGDL4_<lambda_0>_GS,NGDL5_min_GS,NGDL5_avg_GS,NGDL5_<lambda_0>_GS,DGDL1_min_GS,DGDL1_avg_GS,DGDL1_<lambda_0>_GS,DGDL2_min_GS,DGDL2_avg_GS,DGDL2_<lambda_0>_GS,DGDL3_min_GS,DGDL3_avg_GS,DGDL3_<lambda_0>_GS,DGDL4_min_GS,DGDL4_avg_GS,DGDL4_<lambda_0>_GS,DGDL5_min_GS,DGDL5_avg_GS,DGDL5_<lambda_0>_GS,DGDH1_max_GS,DGDH1_avg_GS,DGDH1_<lambda_0>_GS,DGDH2_max_GS,DGDH2_avg_GS,DGDH2_<lambda_0>_GS,DGDH3_max_GS,DGDH3_avg_GS,DGDH3_<lambda_0>_GS,DGDH4_max_GS,DGDH4_avg_GS,DGDH4_<lambda_0>_GS,DGDH5_max_GS,DGDH5_avg_GS,DGDH5_<lambda_0>_GS,PREH_<lambda_0>_GS,PREL_max_GS,PREL_avg_GS,PREL_<lambda_0>_GS
0,ALEXANDER,1996,25.0,4.0,15.306452,11.0,-5.0,2.419355,17.0,1.00,8.862903,12.0,0.0,1.567742,-5.0,-1.048387,10.0,0.0,-6.0,-1.419355,12.0,0.0,-7.0,-1.903226,15.0,0.0,-8.0,-2.403226,16.0,0.0,-9.0,-3.016129,19.0,-2.0,-0.064516,1.0,-3.0,-0.096774,1.0,-4.0,-0.129032,1.0,-5.0,-0.161290,1.0,-6.0,-0.225806,2.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,34.0,-7.0,19.904891,20.0,-12.5,6.788043,25.75,-9.75,13.346467,41.0,1.827174,-12.5,-0.714674,29,-13.5,-0.902174,35,-14.5,-1.130435,43,-15.5,-1.391304,49,-16.5,-1.679348,53,-13.0,-0.230978,9,-14.0,-0.290761,11,-15.0,-0.355978,12,-16.0,-0.432065,14,-17.0,-0.527174,18,8.0,0.649457,41,7.0,0.434783,30,6.0,0.279891,22,5.0,0.165761,13,4.0,0.097826,7,61,0.0,-0.713043,139
1,ALEXANDER,1997,31.0,4.0,17.677419,13.0,-7.5,2.161290,22.0,1.75,9.919355,10.0,0.0,0.896774,-7.5,-0.741935,10.0,0.0,-8.5,-1.096774,11.0,0.0,-9.5,-1.548387,14.0,0.0,-10.5,-2.032258,16.0,0.0,-11.5,-2.645161,19.0,-2.0,-0.064516,1.0,-3.0,-0.096774,1.0,-4.0,-0.129032,1.0,-5.0,-0.177419,2.0,-6.0,-0.258065,3.0,5.0,0.209677,2.0,4.0,0.145161,2.0,3.0,0.096774,1.0,2.0,0.064516,1.0,...,35.0,-3.0,21.527174,19.0,-16.0,6.975543,25.00,-5.75,14.251359,39.2,1.570652,-16.0,-0.527174,28,-17.0,-0.711957,34,-18.0,-0.932065,41,-19.0,-1.171196,45,-20.0,-1.456522,53,-9.0,-0.225543,11,-10.0,-0.304348,15,-11.0,-0.391304,16,-12.0,-0.480978,17,-13.0,-0.576087,18,9.0,1.010870,58,8.0,0.703804,53,7.0,0.434783,37,6.0,0.247283,21,5.0,0.144022,12,56,0.0,-0.734783,142
2,ALEXANDER,1998,28.0,4.0,19.951613,11.0,-2.0,4.241935,18.0,2.00,12.096774,18.8,0.0,2.058065,-2.0,-0.193548,5.0,0.0,-3.0,-0.419355,7.0,0.0,-4.0,-0.741935,10.0,0.0,-5.0,-1.096774,11.0,0.0,-6.0,-1.451613,11.0,-2.0,-0.064516,1.0,-3.0,-0.129032,2.0,-4.0,-0.193548,2.0,-5.0,-0.258065,2.0,-6.0,-0.322581,2.0,2.0,0.064516,1.0,1.0,0.032258,1.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,33.5,2.0,20.997283,19.0,-5.0,7.896739,25.50,0.50,14.447011,39.6,2.818478,-5.0,-0.192935,20,-6.0,-0.328804,25,-7.0,-0.527174,37,-8.0,-0.752717,42,-9.0,-1.013587,48,-4.0,-0.040761,4,-5.0,-0.078804,7,-6.0,-0.144022,12,-7.0,-0.217391,14,-8.0,-0.315217,19,7.5,0.695652,47,6.5,0.451087,34,5.5,0.271739,27,4.5,0.133152,13,3.5,0.067935,6,60,0.0,-0.689130,131
3,

## 4) Merge final datasets (yield + weather features)

In [15]:
# Load restated yields
y_rest = pd.read_csv(DATA_PREP / "yields_restated_muni_year.csv")

ds_station = y_rest.merge(feat_station, on=["Municipality","Year"], how="inner")
ds_reanalysis = y_rest.merge(feat_reanalysis, on=["Municipality","Year"], how="inner")

print("Merged station:", ds_station.shape, "Merged reanalysis:", ds_reanalysis.shape)

ds_station.to_csv(DATA_PREP / "dataset_station_muni_year.csv", index=False)
ds_reanalysis.to_csv(DATA_PREP / "dataset_reanalysis_muni_year.csv", index=False)

display(ds_station.head())


Merged station: (576, 426) Merged reanalysis: (1325, 379)


,Municipality,Year,Yield_restated,Tmax_max_May,Tmax_min_May,Tmax_avg_May,Tmin_max_May,Tmin_min_May,Tmin_avg_May,Tavg_max_May,Tavg_min_May,Tavg_avg_May,P_max_May,P_min_May,P_avg_May,NGDL1_min_May,NGDL1_avg_May,NGDL1_<lambda_0>_May,NGDL2_max_May,NGDL2_min_May,NGDL2_avg_May,NGDL2_<lambda_0>_May,NGDL3_max_May,NGDL3_min_May,NGDL3_avg_May,NGDL3_<lambda_0>_May,NGDL4_max_May,NGDL4_min_May,NGDL4_avg_May,NGDL4_<lambda_0>_May,NGDL5_max_May,NGDL5_min_May,NGDL5_avg_May,NGDL5_<lambda_0>_May,DGDL1_min_May,DGDL1_avg_May,DGDL1_<lambda_0>_May,DGDL2_min_May,DGDL2_avg_May,DGDL2_<lambda_0>_May,DGDL3_min_May,DGDL3_avg_May,DGDL3_<lambda_0>_May,DGDL4_min_May,DGDL4_avg_May,DGDL4_<lambda_0>_May,DGDL5_min_May,DGDL5_avg_May,DGDL5_<lambda_0>_May,DGDH1_max_May,DGDH1_avg_May,DGDH1_<lambda_0>_May,DGDH2_max_May,DGDH2_avg_May,DGDH2_<lambda_0>_May,DGDH3_max_May,DGDH3_avg_May,DGDH3_<lambda_0>_May,DGDH4_max_May,DGDH4_avg_May,...,Tmax_max_GS,Tmax_min_GS,Tmax_avg_GS,Tmin_max_GS,Tmin_min_GS,Tmin_avg_GS,Tavg_max_GS,Tavg_min_GS,Tavg_avg_GS,P_max_GS,P_avg_GS,NGDL1_min_GS,NGDL1_avg_GS,NGDL1_<lambda_0>_GS,NGDL2_min_GS,NGDL2_avg_GS,NGDL2_<lambda_0>_GS,NGDL3_min_GS,NGDL3_avg_GS,NGDL3_<lambda_0>_GS,NGDL4_min_GS,NGDL4_avg_GS,NGDL4_<lambda_0>_GS,NGDL5_min_GS,NGDL5_avg_GS,NGDL5_<lambda_0>_GS,DGDL1_min_GS,DGDL1_avg_GS,DGDL1_<lambda_0>_GS,DGDL2_min_GS,DGDL2_avg_GS,DGDL2_<lambda_0>_GS,DGDL3_min_GS,DGDL3_avg_GS,DGDL3_<lambda_0>_GS,DGDL4_min_GS,DGDL4_avg_GS,DGDL4_<lambda_0>_GS,DGDL5_min_GS,DGDL5_avg_GS,DGDL5_<lambda_0>_GS,DGDH1_max_GS,DGDH1_avg_GS,DGDH1_<lambda_0>_GS,DGDH2_max_GS,DGDH2_avg_GS,DGDH2_<lambda_0>_GS,DGDH3_max_GS,DGDH3_avg_GS,DGDH3_<lambda_0>_GS,DGDH4_max_GS,DGDH4_avg_GS,DGDH4_<lambda_0>_GS,DGDH5_max_GS,DGDH5_avg_GS,DGDH5_<lambda_0>_GS,PREH_<lambda_0>_GS,PREL_max_GS,PREL_avg_GS,PREL_<lambda_0>_GS
0,ALEXANDER,1996,0.907619,25.0,4.0,15.306452,11.0,-5.0,2.419355,17.0,1.00,8.862903,12.0,0.0,1.567742,-5.0,-1.048387,10.0,0.0,-6.0,-1.419355,12.0,0.0,-7.0,-1.903226,15.0,0.0,-8.0,-2.403226,16.0,0.0,-9.0,-3.016129,19.0,-2.0,-0.064516,1.0,-3.0,-0.096774,1.0,-4.0,-0.129032,1.0,-5.0,-0.161290,1.0,-6.0,-0.225806,2.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,34.0,-7.0,19.904891,20.0,-12.5,6.788043,25.75,-9.75,13.346467,41.0,1.827174,-12.5,-0.714674,29,-13.5,-0.902174,35,-14.5,-1.130435,43,-15.5,-1.391304,49,-16.5,-1.679348,53,-13.0,-0.230978,9,-14.0,-0.290761,11,-15.0,-0.355978,12,-16.0,-0.432065,14,-17.0,-0.527174,18,8.0,0.649457,41,7.0,0.434783,30,6.0,0.279891,22,5.0,0.165761,13,4.0,0.097826,7,61,0.0,-0.713043,139
1,ALEXANDER,1997,0.859800,31.0,4.0,17.677419,13.0,-7.5,2.161290,22.0,1.75,9.919355,10.0,0.0,0.896774,-7.5,-0.741935,10.0,0.0,-8.5,-1.096774,11.0,0.0,-9.5,-1.548387,14.0,0.0,-10.5,-2.032258,16.0,0.0,-11.5,-2.645161,19.0,-2.0,-0.064516,1.0,-3.0,-0.096774,1.0,-4.0,-0.129032,1.0,-5.0,-0.177419,2.0,-6.0,-0.258065,3.0,5.0,0.209677,2.0,4.0,0.145161,2.0,3.0,0.096774,1.0,2.0,0.064516,...,35.0,-3.0,21.527174,19.0,-16.0,6.975543,25.00,-5.75,14.251359,39.2,1.570652,-16.0,-0.527174,28,-17.0,-0.711957,34,-18.0,-0.932065,41,-19.0,-1.171196,45,-20.0,-1.456522,53,-9.0,-0.225543,11,-10.0,-0.304348,15,-11.0,-0.391304,16,-12.0,-0.480978,17,-13.0,-0.576087,18,9.0,1.010870,58,8.0,0.703804,53,7.0,0.434783,37,6.0,0.247283,21,5.0,0.144022,12,56,0.0,-0.734783,142
2,ALEXANDER,1998,0.938502,28.0,4.0,19.951613,11.0,-2.0,4.241935,18.0,2.00,12.096774,18.8,0.0,2.058065,-2.0,-0.193548,5.0,0.0,-3.0,-0.419355,7.0,0.0,-4.0,-0.741935,10.0,0.0,-5.0,-1.096774,11.0,0.0,-6.0,-1.451613,11.0,-2.0,-0.064516,1.0,-3.0,-0.129032,2.0,-4.0,-0.193548,2.0,-5.0,-0.258065,2.0,-6.0,-0.322581,2.0,2.0,0.064516,1.0,1.0,0.032258,1.0,0.0,0.000000,0.0,0.0,0.000000,...,33.5,2.0,20.997283,19.0,-5.0,7.896739,25.50,0.50,14.447011,39.6,2.818478,-5.0,-0.192935,20,-6.0,-0.328804,25,-7.0,-0.527174,37,-8.0,-0.752717,42,-9.0,-1.013587,48,-4.0,-0.040761,4,-5.0,-0.078804,7,-6.0,-0.144022,12,-7.0,-0.217391,14,-8.0,-0.315217,19,7.5,0.695652,47,6.5,0.451087,34,5.5,0.271739,27,4.5,0.133152,13,3.5,0.067935,6,60,0.0,-0.6891

## 5) Quick checks

In [16]:
def quick_report(df, name):
    print("\n==", name, "==")
    print("Rows:", len(df), "Municipalities:", df["Municipality"].nunique(), "Years:", df["Year"].nunique())
    print("Missing Yield_restated:", df["Yield_restated"].isna().sum())
    # feature missingness
    feat_cols = [c for c in df.columns if c not in ["Municipality","Year","Yield_restated"]]
    miss_rate = df[feat_cols].isna().mean().sort_values(ascending=False).head(10)
    print("Top missing feature rates:")
    print(miss_rate)

quick_report(ds_station, "dataset_station_muni_year")
quick_report(ds_reanalysis, "dataset_reanalysis_muni_year")



== dataset_station_muni_year ==
Rows: 576 Municipalities: 37 Years: 16
Missing Yield_restated: 0
Top missing feature rates:
PREL_<lambda_0>_Oct     0.010417
PREL_avg_Oct            0.010417
PREL_min_Oct            0.010417
PREL_max_Oct            0.010417
PREH_<lambda_0>_Oct     0.010417
DGDH5_<lambda_0>_Oct    0.010417
DGDH5_avg_Oct           0.010417
DGDH5_max_Oct           0.010417
DGDH4_<lambda_0>_Oct    0.010417
DGDH4_avg_Oct           0.010417
dtype: float64

== dataset_reanalysis_muni_year ==
Rows: 1325 Municipalities: 84 Years: 16
Missing Yield_restated: 0
Top missing feature rates:
Tmax_max_May    0.0
Tmax_min_May    0.0
Tmax_avg_May    0.0
Tmin_max_May    0.0
Tmin_min_May    0.0
Tmin_avg_May    0.0
Tavg_max_May    0.0
Tavg_min_May    0.0
Tavg_avg_May    0.0
P_max_May       0.0
dtype: float64
